# Curriculum Learning for Trading

This notebook demonstrates how to use curriculum learning to progressively train trading models from simple to complex tasks.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.training.curriculum_learner import (
    CurriculumLearner,
    TaskConfig,
    CurriculumTask,
    TaskDifficulty
)
from src.data.binance_fetcher import BinanceFetcher
from src.features.feature_engineering import FeatureEngineer

## Model Definition

Define a simple trading model for demonstration.

In [ ]:
class TradingModel(nn.Module):
    """Simple trading model."""
    def __init__(self, input_dim: int = 10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.network(x)

# Initialize model
model = TradingModel()

## Task Generation

Create data generation functions for different task difficulties.

In [ ]:
def generate_market_data(
    volatility_range: tuple,
    trend_range: tuple,
    time_horizon: int
) -> tuple:
    """Generate synthetic market data."""
    batch_size = 32
    
    # Generate random walk with trend and volatility
    volatility = np.random.uniform(*volatility_range)
    trend = np.random.uniform(*trend_range)
    
    # Generate features
    features = np.random.randn(batch_size, 10)
    
    # Add trend and volatility information
    features[:, 0] = trend
    features[:, 1] = volatility
    
    # Generate targets (optimal positions)
    targets = np.sign(features[:, 0] + np.random.randn(batch_size) * volatility)
    targets = targets.reshape(-1, 1)
    
    return torch.FloatTensor(features), torch.FloatTensor(targets)

# Create task configurations
task_configs = [
    TaskConfig(
        difficulty=TaskDifficulty.VERY_EASY,
        market_volatility=(0.01, 0.02),
        trend_strength=(0.02, 0.03),
        time_horizon=10,
        position_constraints={'max_size': 0.5},
        required_accuracy=0.6,
        sampling_weight=1.0
    ),
    TaskConfig(
        difficulty=TaskDifficulty.EASY,
        market_volatility=(0.02, 0.03),
        trend_strength=(0.015, 0.025),
        time_horizon=20,
        position_constraints={'max_size': 0.7},
        required_accuracy=0.55,
        sampling_weight=0.8
    ),
    TaskConfig(
        difficulty=TaskDifficulty.MEDIUM,
        market_volatility=(0.03, 0.04),
        trend_strength=(0.01, 0.02),
        time_horizon=30,
        position_constraints={'max_size': 1.0},
        required_accuracy=0.5,
        sampling_weight=0.6
    ),
    TaskConfig(
        difficulty=TaskDifficulty.HARD,
        market_volatility=(0.04, 0.05),
        trend_strength=(0.005, 0.015),
        time_horizon=50,
        position_constraints={'max_size': 1.5},
        required_accuracy=0.45,
        sampling_weight=0.4
    )
]

# Create tasks
tasks = [
    CurriculumTask(
        config=config,
        data_generator=generate_market_data,
        performance_metrics=['accuracy', 'profit']
    )
    for config in task_configs
]

## Curriculum Learning

Train the model using curriculum learning.

In [ ]:
# Initialize curriculum learner
learner = CurriculumLearner(
    config={'curriculum': {'scheduler_type': 'performance_based'}},
    model=model,
    tasks=tasks
)

# Initialize optimizer
optimizer = torch.optim.Adam(model.parameters())

# Train with curriculum
num_steps = 1000
history = learner.train(
    num_steps=num_steps,
    optimizer=optimizer,
    eval_frequency=50
)

## Performance Analysis

Analyze the learning progress and task progression.

In [ ]:
def plot_learning_progress(history: dict):
    """Plot learning progress."""
    plt.figure(figsize=(15, 5))
    
    # Plot loss
    plt.subplot(1, 3, 1)
    plt.plot(history['loss'])
    plt.title('Training Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    
    # Plot task difficulty
    plt.subplot(1, 3, 2)
    difficulties = pd.Series(history['task_difficulty'])
    difficulties.value_counts().plot(kind='bar')
    plt.title('Task Distribution')
    plt.xlabel('Task Difficulty')
    plt.ylabel('Count')
    
    # Plot performance
    plt.subplot(1, 3, 3)
    plt.plot(history['performance'])
    plt.title('Performance')
    plt.xlabel('Step')
    plt.ylabel('Accuracy')
    
    plt.tight_layout()
    plt.show()

# Plot progress
plot_learning_progress(history)

# Print curriculum status
status = learner.get_curriculum_status()
print("\nCompleted Tasks:")
for task in status['completed_tasks']:
    print(f"- {task}")

## Task-Specific Analysis

Analyze performance on different task difficulties.

In [ ]:
def evaluate_tasks(model: nn.Module, tasks: list):
    """Evaluate model on each task."""
    results = []
    
    for task in tasks:
        task_results = []
        
        # Evaluate multiple times
        for _ in range(10):
            features, targets = task.generate_data()
            
            with torch.no_grad():
                predictions = model(features)
                metrics = task.evaluate_performance(predictions, targets)
                task_results.append(metrics)
        
        # Average results
        avg_metrics = {}
        for metric in task_results[0].keys():
            avg_metrics[metric] = np.mean([r[metric] for r in task_results])
        
        results.append({
            'difficulty': task.config.difficulty.value,
            **avg_metrics
        })
    
    return pd.DataFrame(results)

# Evaluate tasks
results_df = evaluate_tasks(model, tasks)

# Plot results
plt.figure(figsize=(12, 5))

# Plot accuracy by difficulty
plt.subplot(1, 2, 1)
plt.bar(results_df['difficulty'], results_df['accuracy'])
plt.title('Accuracy by Task Difficulty')
plt.xticks(rotation=45)

# Plot profit by difficulty
plt.subplot(1, 2, 2)
plt.bar(results_df['difficulty'], results_df['profit'])
plt.title('Profit by Task Difficulty')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Print results
print("\nTask Performance:")
print(results_df)

## Model Evaluation

Evaluate the trained model on real market data.

In [ ]:
# Initialize data fetcher
fetcher = BinanceFetcher(config['data']['binance'])

# Fetch test data
test_data = await fetcher.fetch_historical_data()

# Prepare features
engineer = FeatureEngineer(
    input_file='data/raw/BTCUSDT_data.parquet',
    output_dir='data/processed',
    config=config
)

# Generate features
features = await engineer.generate_features()

# Make predictions
with torch.no_grad():
    predictions = model(torch.FloatTensor(features))

# Plot predictions vs actual returns
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(test_data['returns'], predictions.numpy())
plt.xlabel('Actual Returns')
plt.ylabel('Predicted Returns')
plt.title('Predictions vs Actual')

plt.subplot(1, 2, 2)
cumulative_returns = np.cumprod(1 + test_data['returns'])
strategy_returns = np.cumprod(1 + test_data['returns'] * np.sign(predictions.numpy()))

plt.plot(cumulative_returns, label='Buy & Hold')
plt.plot(strategy_returns, label='Strategy')
plt.title('Strategy Performance')
plt.legend()

plt.tight_layout()
plt.show()

## Save Model

Save the trained model and curriculum state.

In [ ]:
# Save curriculum state
learner.save_curriculum_state('models/curriculum_state.pt')
print("Saved curriculum state")